# Fuzzy Classification of PLGA Release Behaviour
### Interpretable, boundary-robust fuzzy classification vs. the crisp baseline

**What this notebook is for.** The project's contribution is *not* a more accurate
model. Two 2026 studies already publish accurate, SHAP-explained models on this exact
dataset (proposal §3.1). The contribution is that the **hard class boundary is
replaced by a graded one**: a formulation just below the cutoff and one just above it
should get *similar, graded* descriptions — not opposite hard labels.

> الهدف من هذا الدفتر ليس دقة أعلى، بل **تفسير أوضح وحدود قرار غير هشة**.
> التركيبة التي تقع تحت الحد بقليل والتي تقع فوقه بقليل يجب أن تُوصفا وصفًا متدرجًا متقاربًا، لا وصفين متعاكسين.

---

## How to read this notebook (the honesty contract)

1. **Every number in the prose below is printed by a code cell above it.** Nothing is
   typed from memory. If a number is not computed here, it is marked `unverified`.
2. **Negative results are reported as results.** Fuzzy encoding is *not* assumed to
   help accuracy (proposal §15). Where it does not help, this notebook says so plainly
   and explains why.
3. **Rules are "leads" until they pass held-out validation** across folds (proposal §9).
   The word "validated" is used only where the test actually passed.
4. **Every metric is read against the ~2.9:1 class imbalance**, never against 50%.

## Structure

| Task | What it does |
|---|---|
| 1 | Load data, fix the `Encapuslation` typo, grouped split by exact Drug SMILES |
| 2 | Crisp class target — AUC of the normalized release curve, cut at 0.5 |
| 3 | **The contribution**: graded (fuzzy) class output around that cutoff |
| 4 | Crisp / fuzzy / hybrid feature representations of the formulation descriptors |
| 5 | XGBoost, three arms × two label types, paired per-fold comparison |
| 6 | **The key evaluation**: boundary robustness |
| 7 | Tree → fuzzy IF–THEN rules, refit inside folds, validated on held-out drugs |
| 8 | SHAP — global importance, crisp-vs-fuzzy feature credit |
| — | Improvement attempts, SELF-CHECK, correction log, open decisions |

---
# Task 0 — Environment

**Why a fixed seed and a single set of folds.** The whole claim of this notebook is a
*comparison* between representations. If the arms saw different folds or different
random seeds, any difference between them could be the split rather than the encoding.
So the folds are computed once and every arm reuses them.

**Note on the two source files named in the brief.** `proposal_v2_detailed.md` and
`membership_functions.ipynb` do not exist under those names in this repository. Their
actual equivalents are used instead and are named explicitly here so the provenance is
auditable:

- `proposal.md` — its own header reads *"Research Proposal (v2 — Detailed)"*. This is
  the authority used for scope and constraints.
- `fuzzypharma/fuzzy.py` — the train-fold-only triangular Low/Med/High membership
  implementation. The functions below are **copied from it, not rewritten**, so this
  notebook stays runnable on its own; the canonical source is that module.

In [1]:
import os, sys, json, warnings, textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, confusion_matrix)
from scipy.stats import binomtest
import xgboost as xgb
import shap

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

SEED = 42
N_SPLITS = 5
np.random.seed(SEED)

FIGDIR = "figures"
os.makedirs(FIGDIR, exist_ok=True)

def savefig(fig, name):
    path = os.path.join(FIGDIR, name)
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print(f"saved {path}")
    return path

print("python     ", sys.version.split()[0])
print("numpy      ", np.__version__)
print("pandas     ", pd.__version__)
print("xgboost    ", xgb.__version__)
print("shap       ", shap.__version__)
print("seed       ", SEED, "| folds", N_SPLITS)

python      3.9.6
numpy       2.0.2
pandas      2.3.3
xgboost     2.1.4
shap        0.49.1
seed        42 | folds 5


/Users/alhanoufalqahtani/Documents/FuzzyPharma/.venv_fp/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
# Task 1 — Load the data and build the grouped split

**Why we group by exact Drug SMILES.** Each drug appears in several formulations. If
formulations of the same drug landed in both the training and the validation fold, the
model could recognise the drug from `Drug MW`/`TPSA`/`LogP` and recite its release
behaviour — that is leakage, and it inflates every score. Grouping on SMILES forces the
model to generalise to a **drug it has never seen** (proposal §8). This is
non-negotiable constraint #1.

**Why SMILES rather than drug name.** There are 89 drug *names* but only 88 unique
SMILES — two names share one recorded structure. Grouping on SMILES is the stricter,
safer choice: it never splits two chemically identical entries across folds.

**Why the typo fix matters.** The source workbook spells the column
`Drug Encapuslation Efficiency`. Silently mis-typing it later would produce a
`KeyError` at best, or a silently dropped feature at worst. We rename it once, here.

**Why we merge two workbooks.** `mp_dataset_initial.xlsx` carries drug identity
(name, SMILES, formulation method, DOI) but not the three computed drug descriptors;
`mp_dataset_processed.xlsx` carries the descriptors. They share `Formulation Index`
row-for-row, so the join is exact, not approximate.

> نقسّم البيانات حسب تركيب الدواء (SMILES) لا حسب الصفوف، حتى لا يتسرّب الدواء نفسه بين التدريب والاختبار.

In [2]:
STATIC = ["Drug MW", "Drug TPSA", "Drug LogP", "Polymer MW", "LA/GA",
          "Initial Drug-to-Polymer Ratio", "Particle Size", "Drug Loading Capacity",
          "Drug Encapsulation Efficiency", "Solubility Enhancer Concentration"]
METHODS = ["O/W", "S/O/W", "W/O/W", "S/W/O/W"]

# Identity / provenance columns. NEVER model inputs (constraint #4).
ID_COLS = ["Formulation Index", "Drug", "Drug SMILES", "DOI"]

raw = pd.read_excel("mp_dataset_initial.xlsx", sheet_name="PLGA_MPs")
print("raw long table:", raw.shape)
print("typo present in source:", "Drug Encapuslation Efficiency" in raw.columns)
raw = raw.rename(columns={"Drug Encapuslation Efficiency": "Drug Encapsulation Efficiency"})
print("after rename     :", "Drug Encapsulation Efficiency" in raw.columns)

proc = pd.read_excel("mp_dataset_processed.xlsx", sheet_name="Sheet1")
desc = proc[["Formulation Index", "Drug MW", "Drug TPSA", "Drug LogP"]].drop_duplicates("Formulation Index")
long = raw.merge(desc, on="Formulation Index", how="left")

print("\nmerged long table:", long.shape)
print("missing values   :", int(long.isna().sum().sum()), "-> no imputation needed (constraint #3)")
print("time unit        : days (decided; proposal §4)")
print("release range    : [%.4f, %.4f]" % (long.Release.min(), long.Release.max()))

raw long table: (4913, 14)
typo present in source: True
after rename     : True



merged long table: (4913, 17)
missing values   : 0 -> no imputation needed (constraint #3)
time unit        : days (decided; proposal §4)
release range    : [0.0000, 1.0816]


## Task 1b — Collapse to one row per formulation, and compute the curve summary

The 4,913 rows are **measurements**, not samples: they are 321 release curves observed
at many time points. Modelling at row level would count a densely-sampled curve 50
times and a sparse one 8 times. So we collapse to **one row per formulation**.

**Why we also compute a *perturbed* AUC here.** To claim later that the crisp boundary
is brittle, we need a perturbation that is smaller than the real experimental
uncertainty. We recompute each curve's AUC from **half of its interior time points**
(keeping both endpoints). This mimics the same experiment reported with a coarser
sampling schedule — a change that should not alter a formulation's class. Task 6 uses
this to count how often the crisp label flips anyway.

In [3]:
def trapz(y, x):
    fn = getattr(np, "trapezoid", None) or np.trapz   # NumPy >= 2 renamed trapz
    return float(fn(y, x))

def normalized_time(t):
    return (t - t.min()) / (t.max() - t.min())

def half_sample_auc(t, r, phase):
    keep = np.zeros(len(t), bool)
    keep[0] = keep[-1] = True
    keep[1:-1][phase::2] = True
    if keep.sum() < 3:
        return np.nan
    tt, rr = t[keep], r[keep]
    return trapz(rr, normalized_time(tt))

rows = []
for fid, g in long.groupby("Formulation Index", sort=True):
    g = g.sort_values("Time").drop_duplicates("Time")
    t = g["Time"].to_numpy(float)
    r = g["Release"].to_numpy(float)
    auc = trapz(r, normalized_time(t))
    hs = [a for a in (half_sample_auc(t, r, p) for p in (0, 1)) if np.isfinite(a)]
    rec = {"Formulation Index": int(fid), "Drug": g["Drug"].iloc[0],
           "Drug SMILES": g["Drug SMILES"].iloc[0],
           "Formulation Method": g["Formulation Method"].iloc[0],
           "DOI": g["DOI"].iloc[0],
           "AUC": auc,
           "AUC half-sample dev": float(np.mean(np.abs(np.array(hs) - auc))) if hs else np.nan,
           "AUC perturbed": hs[0] if hs else auc,
           "N Time Points": len(t),
           "Profile Duration": float(t.max() - t.min()),
           "Max Observed Release": float(r.max())}
    for c in STATIC:
        rec[c] = float(g[c].iloc[0])
    rows.append(rec)

F = pd.DataFrame(rows)
print("formulation-level table:", F.shape)
print("formulations           :", len(F),                 "(expected 321)")
print("unique Drug SMILES     :", F['Drug SMILES'].nunique(), "(expected 88)")
print("unique drug names      :", F['Drug'].nunique(),     "(89 names -> 88 SMILES)")
assert len(F) == 321 and F["Drug SMILES"].nunique() == 88
print("\nprofile duration (days): min %.2f  max %.2f  mean %.2f +/- %.2f"
      % (F["Profile Duration"].min(), F["Profile Duration"].max(),
         F["Profile Duration"].mean(), F["Profile Duration"].std()))

formulation-level table: (321, 21)
formulations           : 321 (expected 321)
unique Drug SMILES     : 88 (expected 88)
unique drug names      : 89 (89 names -> 88 SMILES)

profile duration (days): min 3.00  max 237.73  mean 30.02 +/- 25.47


## Task 1c — GroupKFold and the zero-overlap proof

The assertion below is the proof required by constraint #1. It is not decorative: if a
single SMILES ever appeared in two folds it would raise, and no result downstream would
be trustworthy.

In [4]:
gkf = GroupKFold(n_splits=N_SPLITS)
F["Fold"] = -1
for k, (_, va_idx) in enumerate(gkf.split(F, groups=F["Drug SMILES"])):
    F.loc[F.index[va_idx], "Fold"] = k
assert (F.Fold >= 0).all()

fold_tbl = F.groupby("Fold").agg(formulations=("AUC", "size"),
                                 drugs=("Drug SMILES", "nunique"))
print(fold_tbl.to_string())

# --- the proof: no SMILES appears in two folds ---
sets = {k: set(F.loc[F.Fold == k, "Drug SMILES"]) for k in range(N_SPLITS)}
overlaps = {(a, b): sets[a] & sets[b] for a in range(N_SPLITS) for b in range(a + 1, N_SPLITS)}
worst = max(len(v) for v in overlaps.values())
for (a, b), v in overlaps.items():
    assert not v, f"LEAKAGE: folds {a} and {b} share {len(v)} SMILES"
print(f"\nPAIRWISE SMILES OVERLAP across all {len(overlaps)} fold pairs: max = {worst}")
print("ZERO drug overlap across folds -> assertion PASSED")
print("total SMILES covered:", sum(len(s) for s in sets.values()), "= sum of per-fold uniques (no double count)")

      formulations  drugs
Fold                     
0               65     13
1               64     18
2               64     19
3               64     19
4               64     19

PAIRWISE SMILES OVERLAP across all 10 fold pairs: max = 0
ZERO drug overlap across folds -> assertion PASSED
total SMILES covered: 88 = sum of per-fold uniques (no double count)


---
# Task 2 — The crisp class target (the benchmark's definition)

**What it is.** For each formulation, normalize the time axis to [0, 1], then take the
area under the normalized release curve (AUC). The crisp class is `AUC > 0.5` vs
`AUC <= 0.5`.

**Why the AUC of the whole curve, and not a single release value.** A single reading —
say release at 24 h — describes one instant. Two formulations can match at 24 h and
then diverge completely: one plateaus, the other keeps releasing for two months. The
AUC of the *normalized* curve summarises the **shape of the entire release process**:
a curve that rises early and stays high has a large area, a curve that lags has a small
one. Normalizing time first means a 3-day study and a 238-day study are compared on
profile *shape* rather than on duration.

**What it is NOT.** ⚠️ This is a *mathematical* descriptor of a normalized curve. It is
**not** a clinical burst-release diagnosis (proposal §15). Throughout this notebook the
two classes are called `fast` (AUC > 0.5) and `slow` (AUC <= 0.5) purely as shorthand
for the curve shape.

> AUC هنا وصف رياضي لشكل منحنى التحرر بعد تطبيع الزمن — وليس تشخيصًا دوائيًا.

In [5]:
AUC_THRESHOLD = 0.5
F["y_crisp"] = (F["AUC"] > AUC_THRESHOLD).astype(int)   # 1 = "fast", 0 = "slow"

BASE_RATE = float(F.y_crisp.mean())
IMBALANCE = BASE_RATE / (1 - BASE_RATE)
MAJORITY_BASELINE = max(BASE_RATE, 1 - BASE_RATE)

print("class counts:")
print(F.y_crisp.map({1: "fast (AUC > 0.5)", 0: "slow (AUC <= 0.5)"}).value_counts().to_string())
print(f"\nbase rate (fraction 'fast') : {BASE_RATE:.4f}")
print(f"class imbalance             : {IMBALANCE:.2f} : 1")
print(f"MAJORITY-CLASS BASELINE     : {MAJORITY_BASELINE:.4f}")
print("  ^ read every accuracy below against THIS number, not against 0.50 (constraint #6).")

print(f"\nAUC distribution: min {F.AUC.min():.4f}  q25 {F.AUC.quantile(.25):.4f}  "
      f"median {F.AUC.median():.4f}  q75 {F.AUC.quantile(.75):.4f}  max {F.AUC.max():.4f}")
print(f"max observed Release across all curves: {F['Max Observed Release'].max():.4f}")
print("  ^ exceeds 1.0 and is NOT clipped (constraint #3).")

class counts:
y_crisp
fast (AUC > 0.5)     239
slow (AUC <= 0.5)     82

base rate (fraction 'fast') : 0.7445
class imbalance             : 2.91 : 1
MAJORITY-CLASS BASELINE     : 0.7445
  ^ read every accuracy below against THIS number, not against 0.50 (constraint #6).

AUC distribution: min 0.2482  q25 0.4935  median 0.6143  q75 0.7218  max 0.9930
max observed Release across all curves: 1.0816
  ^ exceeds 1.0 and is NOT clipped (constraint #3).


---
# Task 3 — The fuzzy class output  ⭐ *this is the contribution*

## The problem with the crisp cutoff

`AUC > 0.5` is a step function. A formulation at `AUC = 0.4999` is labelled **slow**;
one at `AUC = 0.5001` is labelled **fast**. They are physically indistinguishable —
the difference between them is smaller than the uncertainty in how the curve was
sampled — yet the crisp label treats them as opposites. This is the "69-vs-70"
brittleness the proposal targets (§1).

## The design

Replace the step with a graded ramp of half-width `w` centred on the cutoff:

$$\mu_{\text{fast}}(a) \;=\; \operatorname{clip}\!\left(\frac{a - (0.5 - w)}{2w},\; 0,\; 1\right),
\qquad \mu_{\text{slow}}(a) = 1 - \mu_{\text{fast}}(a)$$

- Below `0.5 - w` → confidently **slow** (μ = 0).
- Above `0.5 + w` → confidently **fast** (μ = 1).
- Inside the band → a **graded reading**, e.g. `fast 0.55 / slow 0.45`.

The two memberships always sum to 1, so the output reads as a description, not two
unrelated scores.

## How `w` is chosen — and why it is fitted on the training fold only

`w` answers: *how far apart must two AUC values be before we are willing to call them
different classes?* Two things bound it, and **both are computed from training-fold
formulations only** (constraint #2):

1. **A noise floor.** Below this, the data cannot tell the two apart at all. We take
   the median of `|AUC − AUC_perturbed|` over training formulations, where the
   perturbation is the half-sampling from Task 1b. Calling two formulations different
   classes when they differ by less than this is claiming precision the measurement
   does not have.
2. **An interpretability scale.** The noise floor alone turns out to be tiny (printed
   below), so it would grade almost nobody and the fuzzy output would collapse back
   onto the crisp one. We therefore also scale to the spread of the training AUCs:
   `α × IQR(AUC_train)`, with `α = 0.25`.

$$w_{\text{fold}} \;=\; \max\big(\underbrace{\text{median}|\Delta \text{AUC}|}_{\text{noise floor}},\;\; \alpha \cdot \text{IQR}(\text{AUC}_{\text{train}})\big)$$

`α = 0.25` is a **declared design choice, not a fitted optimum** — it is swept in the
Improvement Attempts section, and the effect is reported there. Because `w` depends on
the training AUC distribution, **it differs from fold to fold**, which is exactly the
signature of a train-only fit; the table below shows that.

> عرض الشريط `w` يُحسب من بيانات التدريب فقط: أرضية الضوضاء (أصغر فرق يمكن قياسه) أو ربع المدى الرباعي — أيهما أكبر.

In [6]:
ALPHA_BOUNDARY = 0.25   # declared design choice; swept later

def fit_width(train_df, alpha=ALPHA_BOUNDARY):
    """Fit the fuzzy class-boundary half-width. TRAINING ROWS ONLY."""
    noise_floor = float(train_df["AUC half-sample dev"].median())
    train_iqr   = float(train_df.AUC.quantile(.75) - train_df.AUC.quantile(.25))
    return max(noise_floor, alpha * train_iqr), noise_floor, train_iqr

def mu_fast(auc, w):
    """Graded membership in the 'fast' class. mu_slow = 1 - mu_fast."""
    return np.clip((np.asarray(auc, float) - (0.5 - w)) / (2.0 * w), 0.0, 1.0)

width_rows = []
for k in range(N_SPLITS):
    tr = F[F.Fold != k]
    w, floor, iqr = fit_width(tr)
    width_rows.append({"fold": k, "n_train": len(tr), "noise floor": floor,
                       "train IQR(AUC)": iqr, "alpha*IQR": ALPHA_BOUNDARY * iqr,
                       "w (used)": w, "band": f"[{0.5-w:.3f}, {0.5+w:.3f}]"})
WIDTHS = pd.DataFrame(width_rows)
print("Fuzzy class-boundary width, fitted on each TRAINING fold only:\n")
print(WIDTHS.round(5).to_string(index=False))
print(f"\nw differs across folds ({WIDTHS['w (used)'].min():.4f} .. {WIDTHS['w (used)'].max():.4f})"
      " -> it is a property of the training data, not of the whole dataset.")
print(f"noise floor is {WIDTHS['noise floor'].mean():.4f} on average, i.e. "
      f"{ALPHA_BOUNDARY*WIDTHS['train IQR(AUC)'].mean()/WIDTHS['noise floor'].mean():.0f}x smaller "
      "than the interpretability scale -> the interpretability term is what binds.")

Fuzzy class-boundary width, fitted on each TRAINING fold only:

 fold  n_train  noise floor  train IQR(AUC)  alpha*IQR  w (used)           band
    0      256      0.00473         0.23071    0.05768   0.05768 [0.442, 0.558]
    1      257      0.00530         0.22448    0.05612   0.05612 [0.444, 0.556]
    2      257      0.00494         0.24016    0.06004   0.06004 [0.440, 0.560]
    3      257      0.00461         0.22908    0.05727   0.05727 [0.443, 0.557]
    4      257      0.00442         0.21605    0.05401   0.05401 [0.446, 0.554]

w differs across folds (0.0540 .. 0.0600) -> it is a property of the training data, not of the whole dataset.
noise floor is 0.0048 on average, i.e. 12x smaller than the interpretability scale -> the interpretability term is what binds.


In [7]:
# Assign each formulation the membership computed with the width fitted on the folds
# where it was NOT present (i.e. its own validation-fold width). No leakage.
W_BY_FOLD = WIDTHS.set_index("fold")["w (used)"].to_dict()
F["w"]  = F.Fold.map(W_BY_FOLD)
F["mu_fast"] = mu_fast(F.AUC.to_numpy(), F.w.to_numpy())
F["mu_slow"] = 1 - F["mu_fast"]
F["near_boundary"] = F.AUC.sub(0.5).abs() <= F.w

n_near = int(F.near_boundary.sum())
print(f"formulations receiving a GRADED reading (0 < mu < 1): {n_near} of {len(F)} ({n_near/len(F):.1%})")
print(f"the other {len(F)-n_near} are far enough from the cutoff that fuzzy and crisp agree completely.\n")

show = F.loc[(F.AUC - 0.5).abs().nsmallest(8).index].sort_values("AUC")
show = show.assign(**{
    "crisp says": show.y_crisp.map({1: "FAST", 0: "SLOW"}),
    "fuzzy says": show.mu_fast.map(lambda m: f"fast {m:.2f} / slow {1-m:.2f}")})
print("The eight formulations closest to the cutoff — crisp vs fuzzy reading:\n")
print(show[["Formulation Index", "Drug", "AUC", "crisp says", "fuzzy says"]]
      .round(4).to_string(index=False))

formulations receiving a GRADED reading (0 < mu < 1): 63 of 321 (19.6%)
the other 258 are far enough from the cutoff that fuzzy and crisp agree completely.

The eight formulations closest to the cutoff — crisp vs fuzzy reading:

 Formulation Index                        Drug    AUC crisp says            fuzzy says
               227                     ABT627  0.4936       SLOW fast 0.44 / slow 0.56
               265                 ganciclovir 0.5011       FAST fast 0.51 / slow 0.49
               172  apomorphine hydrochloride  0.5018       FAST fast 0.52 / slow 0.48
               145                 simvastatin 0.5021       FAST fast 0.52 / slow 0.48
               239                     ABT627  0.5023       FAST fast 0.52 / slow 0.48
               309 thioridazine hydrochloride  0.5042       FAST fast 0.53 / slow 0.47
               200                fenretinide  0.5054       FAST fast 0.55 / slow 0.45
               136                  prilocaine 0.5064       FAST fast 0.55 

### Why this fixes the brittleness

Look at the table above. Formulations either side of 0.5 differ in AUC by a few
thousandths — far less than the measurement's own resolution — yet the **crisp** column
prints two different words in capital letters, with no hint that the call was a
coin-flip. The **fuzzy** column prints readings near `0.50 / 0.50`, which carries its
own warning: *this formulation sits on the fence; do not act on the label alone.*

That is the whole point. The fuzzy output does not pretend to know more than the crisp
one — it declines to pretend to know **more than the data supports**.

---
# Task 4 — Feature representations: crisp, fuzzy, hybrid

## What goes in — and what is deliberately kept out

**Time is NOT a feature.** In the team's completed study, time had mean |SHAP| ≈ 0.219,
about **six times** the next feature. Any fuzzy signal from formulation descriptors
would be swamped by it, and rules pairing a formulation condition with a time condition
get credited for an effect time alone explains. Per proposal §7, time enters this task
**through the target definition** (the AUC of the curve over time) and nowhere else.
This is constraint #5.

**Identity columns are never inputs** (constraint #4): `Formulation Index`, `Drug`,
`Drug SMILES`, `DOI` are used for grouping and reporting only.

## The two degenerate features — and why they get a reduced treatment

`LA/GA` and `Solubility Enhancer Concentration` are each dominated by one repeated
value (printed below). Splitting them into Low/Medium/High is not just uninformative,
it is **misleading**: the 25th, 50th and 75th percentiles all land on the same number,
so "Low", "Medium" and "High" would describe the same physical value.

Instead each gets a **two-level indicator** — *is it the dominant value, or not?* — with
the dominant value determined **on the training fold**. This is honest and it is also
physically readable: for `Solubility Enhancer Concentration` the dominant value is 0,
so the indicator literally means *"no solubility enhancer"*; for `LA/GA` the dominant
value is 1.0, meaning *"50:50 PLGA"*.

## The three arms

| Arm | Columns |
|---|---|
| **crisp** | 10 raw descriptors + 4 formulation-method one-hots |
| **fuzzy** | Low/Med/High memberships for the 8 fuzzifiable descriptors + 2 degenerate indicators + 4 method one-hots |
| **hybrid** | crisp ∪ fuzzy |

The membership code below is **copied from `fuzzypharma/fuzzy.py`**, not rewritten —
including its fallback that recomputes knots over *distinct* values when raw quantiles
tie. Knots are fitted on the training fold only.

In [8]:
# ---- degeneracy audit (whole dataset, for reporting only; the fit is per-fold) ----
deg_rows = []
for c in STATIC:
    vc = F[c].value_counts(normalize=True)
    deg_rows.append({"feature": c, "n unique": F[c].nunique(),
                     "most common value": vc.index[0], "its share": vc.iloc[0]})
DEG = pd.DataFrame(deg_rows).sort_values("its share", ascending=False)
print("Degeneracy audit — how much of each feature is a single repeated value:\n")
print(DEG.round(4).to_string(index=False))

DEGENERATE = ["LA/GA", "Solubility Enhancer Concentration"]
FUZZABLE = [c for c in STATIC if c not in DEGENERATE]
print(f"\nreduced (two-level) treatment : {DEGENERATE}")
print(f"full Low/Med/High treatment   : {len(FUZZABLE)} features")

Degeneracy audit — how much of each feature is a single repeated value:

                          feature  n unique  most common value  its share
                            LA/GA         6             1.0000     0.7072
Solubility Enhancer Concentration        17             0.0000     0.6355
    Initial Drug-to-Polymer Ratio        89             0.2500     0.2181
                        Drug TPSA        80            94.8300     0.1869
                          Drug MW        87           392.4670     0.1526
                        Drug LogP        88             1.8957     0.1526
                       Polymer MW        56            12.0000     0.1277
                    Particle Size       267            35.0000     0.0405
            Drug Loading Capacity       283             8.6000     0.0125
    Drug Encapsulation Efficiency       294            90.9565     0.0093

reduced (two-level) treatment : ['LA/GA', 'Solubility Enhancer Concentration']
full Low/Med/High treatment   : 8

In [9]:
# ============================================================================
#  Membership functions — COPIED VERBATIM IN BEHAVIOUR from fuzzypharma/fuzzy.py
#  (reproduced here so this notebook runs standalone; that module is canonical)
# ============================================================================
LABELS = {2: ("Low", "High"), 3: ("Low", "Medium", "High"),
          5: ("VeryLow", "Low", "Medium", "High", "VeryHigh")}

def fit_knots(values, n_sets=3, strategy="quantile"):
    """Learn knots for one feature, degrading gracefully on discrete data."""
    v = np.asarray(values, float); v = v[np.isfinite(v)]
    levels = np.linspace(0.25, 0.75, n_sets)
    make = (lambda a: np.quantile(a, levels)) if strategy == "quantile" \
           else (lambda a: np.linspace(a.min(), a.max(), n_sets))
    knots = make(v)
    if np.all(np.diff(knots) > 0):
        return knots, strategy
    uniq = np.unique(v)                      # raw quantiles tied -> retry on distinct values
    if uniq.size >= n_sets:
        knots = make(uniq)
        if np.all(np.diff(knots) > 0):
            return knots, strategy + "-unique"
    return None, "skipped"

def triangular_memberships(x, knots):
    """Shoulder-triangle memberships; each row sums to 1 (a Ruspini partition)."""
    x = np.asarray(x, float)[:, None]; k = np.asarray(knots, float); n = len(k)
    out = np.zeros((x.shape[0], n))
    out[:, 0]  = np.clip((k[1] - x[:, 0]) / (k[1] - k[0]), 0, 1)          # left shoulder
    out[:, -1] = np.clip((x[:, 0] - k[-2]) / (k[-1] - k[-2]), 0, 1)       # right shoulder
    for i in range(1, n - 1):
        rising  = (x[:, 0] - k[i-1]) / (k[i] - k[i-1])
        falling = (k[i+1] - x[:, 0]) / (k[i+1] - k[i])
        out[:, i] = np.clip(np.minimum(rising, falling), 0, 1)
    return out

class Fuzzifier:
    """Fit knots + degenerate-feature modes on TRAINING rows only, then apply."""
    def __init__(self, n_sets=3, strategy="quantile"):
        self.n_sets, self.strategy = n_sets, strategy
    def fit(self, train_df):
        self.labels = LABELS[self.n_sets]
        self.knots = {c: fit_knots(train_df[c].to_numpy(), self.n_sets, self.strategy)
                      for c in FUZZABLE}
        self.mode  = {c: float(train_df[c].mode().iloc[0]) for c in DEGENERATE}
        return self
    def transform(self, X):
        block = {}
        for c in FUZZABLE:
            knots, _ = self.knots[c]
            if knots is None:
                continue
            for lab, col in zip(self.labels, triangular_memberships(X[c].to_numpy(), knots).T):
                block[f"{c} is {lab}"] = col
        for c in DEGENERATE:
            block[f"{c} is {self.mode[c]:g}"] = (X[c].to_numpy() == self.mode[c]).astype(float)
        return pd.DataFrame(block, index=X.index)
    def report(self):
        return pd.DataFrame([{"feature": c, "knot source": self.knots[c][1],
                              "knots": None if self.knots[c][0] is None
                                       else np.round(self.knots[c][0], 4).tolist()}
                             for c in FUZZABLE])

def methods_block(X):
    return pd.DataFrame({f"Method={m}": (X["Formulation Method"] == m).astype(float).to_numpy()
                         for m in METHODS}, index=X.index)

def crisp_block(X):
    return pd.concat([X[STATIC].astype(float), methods_block(X)], axis=1)

def build_arm(arm, fz, X):
    if arm == "crisp":  return crisp_block(X)
    if arm == "fuzzy":  return pd.concat([fz.transform(X), methods_block(X)], axis=1)
    if arm == "hybrid": return pd.concat([crisp_block(X), fz.transform(X)], axis=1)
    raise ValueError(arm)

print("membership machinery defined.")

membership machinery defined.


In [10]:
# Show the knots actually learned, and prove they are fold-specific
_fz0 = Fuzzifier(3, "quantile").fit(F[F.Fold != 0])
print("Knots learned on fold-0 TRAINING data (Low / Medium / High peak locations):\n")
print(_fz0.report().to_string(index=False))
print("\ndegenerate-feature dominant values (from training fold):", _fz0.mode)

print("\nFeature-block widths (fold 0):")
for arm in ("crisp", "fuzzy", "hybrid"):
    blk = build_arm(arm, _fz0, F[F.Fold != 0])
    print(f"  {arm:7s}: {blk.shape[1]:3d} columns")

print("\nSanity — memberships form a partition (each row sums to 1) for a fuzzifiable feature:")
_m = triangular_memberships(F["Particle Size"].to_numpy(), _fz0.knots["Particle Size"][0])
print("  Particle Size membership row sums: min %.6f max %.6f" % (_m.sum(1).min(), _m.sum(1).max()))

Knots learned on fold-0 TRAINING data (Low / Medium / High peak locations):

                      feature knot source                       knots
                      Drug MW    quantile [303.789, 371.171, 510.631]
                    Drug TPSA    quantile   [39.2725, 71.415, 102.28]
                    Drug LogP    quantile     [2.1876, 3.5904, 4.694]
                   Polymer MW    quantile          [12.0, 34.0, 46.0]
Initial Drug-to-Polymer Ratio    quantile            [0.1, 0.2, 0.25]
                Particle Size    quantile     [16.0987, 32.4, 62.675]
        Drug Loading Capacity    quantile   [4.2994, 9.4655, 17.3198]
Drug Encapsulation Efficiency    quantile      [48.1, 71.7875, 85.82]

degenerate-feature dominant values (from training fold): {'LA/GA': 1.0, 'Solubility Enhancer Concentration': 0.0}

Feature-block widths (fold 0):
  crisp  :  14 columns
  fuzzy  :  30 columns
  hybrid :  40 columns

Sanity — memberships form a partition (each row sums to 1) for a fuzzifiable

---
# Task 5 — Train and compare: crisp vs fuzzy vs hybrid

## What is compared

Three **feature** representations × two **label** types = six arms, all on identical
folds and identical seeds:

- **hard labels** — the crisp 0/1 class from Task 2.
- **soft labels** — the graded membership μ_fast from Task 3. XGBoost's
  `binary:logistic` objective accepts probability-valued labels, so the *model
  architecture is byte-for-byte identical* between the two; only the target differs.
  This makes it a clean test of the graded target rather than a confound.

## Why paired, per-fold comparison

Folds differ in difficulty: one fold's drugs may be far easier than another's. That
between-fold variance is large and it lands in the ± sd of a mean, where it drowns the
much smaller between-arm difference. Reporting `0.70 ± 0.06` vs `0.72 ± 0.06` makes two
arms look indistinguishable even when one wins on **every single fold**. So we report
the **per-fold difference** and a **win count** as well (proposal §12).

## How to read accuracy here

⚠️ The majority-class baseline is printed above. An "accuracy of 0.70" on a 2.9:1
problem is **worse than always guessing 'fast'**. Read the table with that in mind;
`balanced accuracy` and `AUROC` are the metrics that are not fooled by the imbalance.

> المقارنة تتم على نفس الطيّات ونفس البذرة العشوائية، ونقارن الفروق طيّة بطيّة لأن صعوبة الطيّة تخفي الفروق الحقيقية.

In [11]:
XGB_PARAMS = dict(objective="binary:logistic", eval_metric="logloss", max_depth=3,
                  eta=0.1, subsample=0.9, colsample_bytree=0.9, min_child_weight=3,
                  reg_lambda=1.0, seed=SEED, nthread=1)
N_ROUNDS = 250

def fit_predict(Xtr, ytr, Xva):
    dtr = xgb.DMatrix(Xtr.to_numpy(float), label=np.asarray(ytr, float),
                      feature_names=list(Xtr.columns))
    dva = xgb.DMatrix(Xva.to_numpy(float), feature_names=list(Xva.columns))
    bst = xgb.train(XGB_PARAMS, dtr, num_boost_round=N_ROUNDS)
    return bst, bst.predict(dva)

def score(y, p):
    yh = (p > 0.5).astype(int)
    return dict(accuracy=accuracy_score(y, yh),
                balanced_acc=balanced_accuracy_score(y, yh),
                precision=precision_score(y, yh, zero_division=0),
                recall=recall_score(y, yh, zero_division=0),
                f1=f1_score(y, yh, zero_division=0),
                auroc=roc_auc_score(y, p) if len(np.unique(y)) > 1 else np.nan)

def run_cv(n_sets=3, strategy="quantile", alpha=ALPHA_BOUNDARY, collect=False):
    recs, preds, cms = [], [], {}
    for k in range(N_SPLITS):
        tr, va = F[F.Fold != k], F[F.Fold == k]
        w, _, _ = fit_width(tr, alpha)                 # TRAIN ONLY
        fz = Fuzzifier(n_sets, strategy).fit(tr)       # TRAIN ONLY
        y_hard, y_soft = tr.y_crisp.to_numpy(), mu_fast(tr.AUC.to_numpy(), w)
        y_va = va.y_crisp.to_numpy()
        for arm in ("crisp", "fuzzy", "hybrid"):
            Xtr, Xva = build_arm(arm, fz, tr), build_arm(arm, fz, va)
            for lab, ytr in (("hard", y_hard), ("soft", y_soft)):
                _, p = fit_predict(Xtr, ytr, Xva)
                recs.append(dict(fold=k, arm=arm, label=lab, **score(y_va, p)))
                if collect:
                    preds.append(pd.DataFrame(dict(
                        fold=k, arm=arm, label=lab,
                        fid=va["Formulation Index"].to_numpy(), AUC=va.AUC.to_numpy(),
                        y=y_va, p=p, mu=mu_fast(va.AUC.to_numpy(), w), w=w)))
                    cms.setdefault((arm, lab), np.zeros((2, 2), int))
                    cms[(arm, lab)] += confusion_matrix(y_va, (p > 0.5).astype(int), labels=[0, 1])
    out = {"metrics": pd.DataFrame(recs)}
    if collect:
        out["preds"] = pd.concat(preds, ignore_index=True)
        out["cms"] = cms
    return out

RES = run_cv(collect=True)
METRICS, PREDS, CMS = RES["metrics"], RES["preds"], RES["cms"]

SUMMARY = (METRICS.groupby(["arm", "label"])
           [["accuracy", "balanced_acc", "precision", "recall", "f1", "auroc"]]
           .mean().round(4))
print(f"MAJORITY-CLASS BASELINE accuracy = {MAJORITY_BASELINE:.4f}   (imbalance {IMBALANCE:.2f}:1)")
print("=" * 78)
print(SUMMARY.to_string())
print("=" * 78)
below = SUMMARY[SUMMARY.accuracy < MAJORITY_BASELINE]
print(f"\n>>> {len(below)} of {len(SUMMARY)} arms score BELOW the majority-class baseline on accuracy.")
print(">>> Accuracy is therefore not evidence of skill here. Use balanced accuracy / AUROC.")

MAJORITY-CLASS BASELINE accuracy = 0.7445   (imbalance 2.91:1)
              accuracy  balanced_acc  precision  recall      f1   auroc
arm    label                                                           
crisp  hard     0.7004        0.4837     0.7370  0.9169  0.8126  0.5583
       soft     0.7098        0.5007     0.7425  0.9258  0.8196  0.6026
fuzzy  hard     0.6848        0.4844     0.7366  0.8869  0.8007  0.6090
       soft     0.7067        0.5168     0.7540  0.8881  0.8108  0.5982
hybrid hard     0.7037        0.5166     0.7570  0.8972  0.8141  0.5742
       soft     0.7254        0.5335     0.7578  0.9260  0.8273  0.6047

>>> 6 of 6 arms score BELOW the majority-class baseline on accuracy.
>>> Accuracy is therefore not evidence of skill here. Use balanced accuracy / AUROC.


In [12]:
def paired(metric):
    piv = METRICS.pivot_table(index="fold", columns=["arm", "label"], values=metric)
    ref = piv[("crisp", "hard")]
    rows = []
    for col in piv.columns:
        if col == ("crisp", "hard"):
            continue
        d = piv[col] - ref
        rows.append({"arm": col[0], "label": col[1], "mean diff": d.mean(),
                     "wins /5": int((d > 0).sum()),
                     "per-fold diff": np.round(d.to_numpy(), 3).tolist()})
    return pd.DataFrame(rows)

for m in ("accuracy", "auroc", "balanced_acc"):
    print(f"\n=== PAIRED per-fold {m} vs the crisp/hard reference ===")
    print(paired(m).round(4).to_string(index=False))
print("\nReading guide: 'wins /5' of 0-1 or 4-5 is a consistent direction;")
print("2-3 is what noise looks like on 5 folds.")


=== PAIRED per-fold accuracy vs the crisp/hard reference ===
   arm label  mean diff  wins /5                       per-fold diff
 crisp  soft     0.0094        2       [0.0, 0.0, 0.016, 0.031, 0.0]
 fuzzy  hard    -0.0157        2 [0.015, -0.062, -0.078, 0.047, 0.0]
 fuzzy  soft     0.0062        1      [0.0, -0.062, 0.0, 0.094, 0.0]
hybrid  hard     0.0032        1   [-0.031, 0.0, -0.078, 0.125, 0.0]
hybrid  soft     0.0250        2       [0.0, 0.0, 0.031, 0.094, 0.0]

=== PAIRED per-fold auroc vs the crisp/hard reference ===
   arm label  mean diff  wins /5                        per-fold diff
 crisp  soft     0.0443        5  [0.018, 0.057, 0.112, 0.033, 0.002]
 fuzzy  hard     0.0506        5   [0.079, 0.024, 0.054, 0.026, 0.07]
 fuzzy  soft     0.0398        4   [0.056, -0.005, 0.1, 0.033, 0.015]
hybrid  hard     0.0158        4    [0.038, 0.025, 0.0, 0.004, 0.013]
hybrid  soft     0.0463        4 [0.044, -0.013, 0.127, 0.052, 0.022]

=== PAIRED per-fold balanced_acc vs the cris

### One result that does favour fuzzy — reported, with its limits

Honesty cuts both ways: a result that favours the fuzzy side must not be omitted
either. Read the AUROC table above. Against the crisp/hard reference, the arms using
the **graded (soft) target** and/or the **fuzzy features** win on **4 or 5 folds out of
5**, with a mean AUROC gain of roughly **+0.04 to +0.05**. Unlike the accuracy column —
where win counts sit at 1-2 of 5, i.e. noise — this direction is *consistent across
folds*, which is exactly the signal the paired design exists to detect.

**Why it is still reported as a lead and not a win:**

1. **It is a ranking gain, not a decision gain.** AUROC measures whether the model
   orders formulations correctly; it never has to commit to a label. The same arms do
   **not** rise above the majority-class baseline on accuracy. A better ranking that
   still yields no usable decision rule is a modest result.
2. **The regime is weak.** AUROC moves from ~0.56 to ~0.61. Both numbers are close
   enough to 0.50 that the gain is a small improvement on a barely-working model — the
   same pattern proposal §14 flagged, where "fuzzification made a failing model fail
   less".
3. **n is small.** Each fold's AUROC is computed on ~64 formulations, so a 0.04
   difference is well within what 5 folds can produce by chance even when the sign is
   consistent.

So: a consistent, small, ranking-only gain. Worth stating; not worth claiming as the
contribution. The contribution is Task 6.

> فرق AUROC صغير لكنه ثابت الاتجاه عبر الطيّات — نذكره بأمانة، ولا نبني عليه ادعاءً.

In [13]:
print("Pooled confusion matrices (rows = true, cols = predicted; 0 = slow, 1 = fast)\n")
for key in [("crisp", "hard"), ("hybrid", "soft")]:
    cm = CMS[key]
    print(f"--- {key[0]} / {key[1]} labels ---")
    print(pd.DataFrame(cm, index=["true slow", "true fast"],
                       columns=["pred slow", "pred fast"]).to_string())
    tn, fp, fn, tp = cm.ravel()
    print(f"    of {tn+fp} truly-slow formulations, {tn} were caught "
          f"({tn/(tn+fp):.1%} specificity) -> the minority class is where the models fail.\n")

Pooled confusion matrices (rows = true, cols = predicted; 0 = slow, 1 = fast)

--- crisp / hard labels ---
           pred slow  pred fast
true slow          5         77
true fast         19        220
    of 82 truly-slow formulations, 5 were caught (6.1% specificity) -> the minority class is where the models fail.

--- hybrid / soft labels ---
           pred slow  pred fast
true slow         11         71
true fast         17        222
    of 82 truly-slow formulations, 11 were caught (13.4% specificity) -> the minority class is where the models fail.



In [14]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
order = [("crisp", "hard"), ("crisp", "soft"), ("fuzzy", "hard"),
         ("fuzzy", "soft"), ("hybrid", "hard"), ("hybrid", "soft")]
names = [f"{a}\n{l}" for a, l in order]
cols = ["#4C78A8" if l == "hard" else "#F58518" for a, l in order]

acc = [SUMMARY.loc[o, "accuracy"] for o in order]
axes[0].bar(names, acc, color=cols)
axes[0].axhline(MAJORITY_BASELINE, ls="--", c="crimson", lw=2,
                label=f"majority baseline {MAJORITY_BASELINE:.3f}")
axes[0].set_ylim(0.5, 0.82); axes[0].set_ylabel("accuracy")
axes[0].set_title("Accuracy — every arm is at or below the baseline")
axes[0].legend(fontsize=8)

auc_ = [SUMMARY.loc[o, "auroc"] for o in order]
axes[1].bar(names, auc_, color=cols)
axes[1].axhline(0.5, ls="--", c="crimson", lw=2, label="chance 0.500")
axes[1].set_ylim(0.4, 0.72); axes[1].set_ylabel("AUROC")
axes[1].set_title("AUROC — weak but above chance")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.tick_params(axis="x", labelsize=8); ax.grid(axis="y", alpha=.3)
fig.suptitle("Task 5 — crisp vs fuzzy vs hybrid, identical folds and seeds", fontweight="bold")
fig.tight_layout()
savefig(fig, "fig1_arm_comparison.png"); plt.close(fig)

saved figures/fig1_arm_comparison.png


---
# Task 6 — Boundary robustness  ⭐ *the key evaluation*

Task 5 measured accuracy — which is **not** what this project claims to improve. This
section measures what it does claim: that the fuzzy output degrades **gracefully** at
the class boundary where the crisp output degrades **catastrophically**.

## The test

Take each formulation and recompute its AUC from **half its interior time points**
(Task 1b). This is a perturbation of the *reporting*, not of the *formulation* — the
same microparticles, a coarser sampling schedule. A trustworthy class assignment should
barely move.

- For the **crisp** label the outcome is binary: it either stays or **flips 0 ↔ 1**, a
  change of magnitude 1.0 — the maximum possible.
- For the **fuzzy** membership the change is proportional to how far the AUC actually
  moved, bounded by `|ΔAUC| / 2w`.

We report the flip count, the mean absolute change under each scheme, and the
sensitivity (label change per unit AUC) at the boundary.

In [15]:
a0 = F.AUC.to_numpy()
a1 = F["AUC perturbed"].to_numpy()
w_arr = F.w.to_numpy()

crisp_delta = np.abs((a0 > 0.5).astype(float) - (a1 > 0.5).astype(float))
fuzzy_delta = np.abs(mu_fast(a0, w_arr) - mu_fast(a1, w_arr))
nb = F.near_boundary.to_numpy()

N_FLIPS = int(crisp_delta.sum())
print("PERTURBATION: AUC recomputed from half the interior time points")
print(f"  median |delta AUC| induced      : {np.median(np.abs(a0-a1)):.5f}  (tiny — a reporting change, not a formulation change)")
print()
print(f"  CRISP label flips               : {N_FLIPS} / {len(F)} formulations ({N_FLIPS/len(F):.2%})")
print(f"  CRISP mean |change|             : {crisp_delta.mean():.4f}")
print(f"  FUZZY mean |change|             : {fuzzy_delta.mean():.4f}")
print(f"  FUZZY max  |change|             : {fuzzy_delta.max():.4f}")
print(f"  ratio (crisp / fuzzy mean)      : {crisp_delta.mean()/max(fuzzy_delta.mean(),1e-12):.2f}x")
print()
print(f"  Restricted to the {int(nb.sum())} near-boundary formulations:")
print(f"    CRISP flips  : {int(crisp_delta[nb].sum())} / {int(nb.sum())} ({crisp_delta[nb].mean():.1%})")
print(f"    FUZZY mean |change| : {fuzzy_delta[nb].mean():.4f}")
print()
if N_FLIPS:
    print(f"  On the {N_FLIPS} formulations that DID flip, the crisp label changed by 1.000")
    print(f"  while the fuzzy membership changed by only {fuzzy_delta[crisp_delta>0].mean():.4f} on average.")
    print("  Same evidence, same perturbation — one output screams, the other shrugs.")
print()
W_MEAN = float(WIDTHS["w (used)"].mean())
print("SENSITIVITY at the cutoff (label change per unit AUC):")
print(f"  crisp : infinite (a step at 0.5 — an arbitrarily small AUC change flips the label)")
print(f"  fuzzy : 1/(2w) = {1/(2*W_MEAN):.2f}  (bounded, so the output can never move faster than the evidence)")

PERTURBATION: AUC recomputed from half the interior time points
  median |delta AUC| induced      : 0.00346  (tiny — a reporting change, not a formulation change)

  CRISP label flips               : 4 / 321 formulations (1.25%)
  CRISP mean |change|             : 0.0125
  FUZZY mean |change|             : 0.0077
  FUZZY max  |change|             : 0.2068
  ratio (crisp / fuzzy mean)      : 1.62x

  Restricted to the 63 near-boundary formulations:
    CRISP flips  : 4 / 63 (6.3%)
    FUZZY mean |change| : 0.0348

  On the 4 formulations that DID flip, the crisp label changed by 1.000
  while the fuzzy membership changed by only 0.0401 on average.
  Same evidence, same perturbation — one output screams, the other shrugs.

SENSITIVITY at the cutoff (label change per unit AUC):
  crisp : infinite (a step at 0.5 — an arbitrarily small AUC change flips the label)
  fuzzy : 1/(2w) = 8.77  (bounded, so the output can never move faster than the evidence)


In [16]:
best = PREDS[(PREDS.arm == "hybrid") & (PREDS.label == "soft")].copy()
best["near"] = best.AUC.sub(0.5).abs() <= best.w
nbm = best[best.near]
mae_mu    = float(np.abs(nbm.p - nbm.mu).mean())
mae_crisp = float(np.abs(nbm.p - nbm.y).mean())
print(f"Model side — the {len(nbm)} near-boundary validation formulations (hybrid/soft arm):")
print(f"  MAE( model output , graded membership mu ) = {mae_mu:.4f}")
print(f"  MAE( model output , hard crisp label     ) = {mae_crisp:.4f}")
print(f"\nThe model's own output is {mae_crisp-mae_mu:.4f} closer to the graded target than to the")
print("hard one. Near the boundary the model is genuinely uncertain; the graded target is a")
print("more faithful description of what it knows than a forced 0 or 1.")

Model side — the 63 near-boundary validation formulations (hybrid/soft arm):
  MAE( model output , graded membership mu ) = 0.3866
  MAE( model output , hard crisp label     ) = 0.5128

The model's own output is 0.1262 closer to the graded target than to the
hard one. Near the boundary the model is genuinely uncertain; the graded target is a
more faithful description of what it knows than a forced 0 or 1.


In [17]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

# --- left: step vs smooth ---
grid = np.linspace(0.25, 0.75, 800)
ax = axes[0]
ax.plot(grid, (grid > 0.5).astype(float), lw=2.5, c="#D62728",
        label="crisp: hard step at AUC = 0.5", drawstyle="steps-post")
ax.plot(grid, mu_fast(grid, W_MEAN), lw=2.5, c="#1F77B4",
        label=f"fuzzy: graded ramp (w = {W_MEAN:.3f})")
ax.axvspan(0.5 - W_MEAN, 0.5 + W_MEAN, color="#1F77B4", alpha=.10)
ax.axvline(0.5, c="k", lw=.8, ls=":")
sub = F[(F.AUC > 0.25) & (F.AUC < 0.75)]
ax.scatter(sub.AUC, sub.mu_fast, s=14, c="#1F77B4", alpha=.55, zorder=5,
           label=f"{len(sub)} real formulations")
ax.scatter(sub.AUC, sub.y_crisp, s=14, marker="x", c="#D62728", alpha=.45, zorder=5)
ax.set_xlabel("AUC of the normalized release curve")
ax.set_ylabel(r"assignment to class 'fast'")
ax.set_title("Class assignment: step vs graded")
ax.legend(fontsize=8, loc="upper left"); ax.grid(alpha=.3)

# --- right: change under perturbation ---
ax = axes[1]
ax.scatter(np.abs(a0 - a1), crisp_delta, s=22, marker="x", c="#D62728",
           alpha=.7, label=f"crisp change ({N_FLIPS} flips of size 1.0)")
ax.scatter(np.abs(a0 - a1), fuzzy_delta, s=16, c="#1F77B4", alpha=.6,
           label="fuzzy membership change")
ax.set_xscale("symlog", linthresh=1e-4)
ax.set_xlabel(r"|$\Delta$AUC| induced by the perturbation")
ax.set_ylabel("resulting change in class assignment")
ax.set_title("Response to a perturbation smaller than the measurement")
ax.legend(fontsize=8); ax.grid(alpha=.3)

fig.suptitle("Task 6 — boundary robustness: the actual contribution", fontweight="bold")
fig.tight_layout()
savefig(fig, "fig2_boundary_robustness.png"); plt.close(fig)

saved figures/fig2_boundary_robustness.png


## What this means for a pharmacist (plain language)

Imagine screening formulations and the model reports a class.

**With the crisp model** every formulation comes back as one word: *fast* or *slow*.
Two formulations whose release curves are, for practical purposes, identical can come
back with opposite words. Nothing in the output tells you which calls were confident
and which were coin-flips. If you re-ran one of those experiments with readings taken
on a slightly different schedule, its label could change — and the number of
formulations to which that applies is printed above.

**With the fuzzy model** those same borderline formulations come back as
*"fast 0.52 / slow 0.48"*. That reading is doing two jobs: it still tells you which side
it leans, **and** it warns you the lean is weak. You would put those formulations in a
"needs a confirmatory release study" pile instead of acting on a label that a coarser
sampling schedule could have reversed. The formulations far from the cutoff still come
back as a clean *fast 1.00 / slow 0.00* — the fuzzy model only hedges where hedging is
warranted.

**What it does not buy you.** It does not make the model more accurate (Task 5), and it
does not make an uninformative feature informative. It changes *how the answer is
reported*, not *how much the model knows*.

> بالنسبة للصيدلي: المخرج المتدرج يقول "أميل إلى سريع لكن بثقة ضعيفة" بدل كلمة قاطعة قد تنقلب لو أُعيد القياس بجدول أخذ عينات مختلف.

---
# Task 7 — Tree → fuzzy IF–THEN rules

## Starting point and what was wrong with it

The collaborator's depth-3 `DecisionTreeRegressor` on
`[Time, Polymer MW, Particle Size, LA/GA]` produced 8 rules. Two problems make those
**exploratory only**, exactly as proposal §9 states:

1. It was fitted on the **full dataset**, so its thresholds already encode the
   held-out drugs. Nothing measured on it is a generalisation estimate.
2. It used **Time** as a splitting feature, and most of its splits went to Time — so
   the rules describe *when you looked at the curve*, not *what the formulation is*
   (proposal §7). It never used `LA/GA` at all.

## What is done here instead

- The tree is refit **inside each training fold** (5 independent trees).
- It splits on **formulation descriptors only** — no Time.
- Its crisp thresholds are converted to fuzzy antecedents using **that fold's own
  membership functions** from Task 4 (the published tree→fuzzy method, proposal §3.2):
  a split `f <= t` becomes `f is {the sets at or below where t sits}`, whose degree is
  the sum of those memberships. Because the memberships form a partition summing to 1,
  this is a smooth, monotone fuzzy version of the crisp test.
- Rule firing strength = **min** over antecedents (Mamdani AND).

## The validation test — and why it is strict

A rule is applied to the **held-out drugs** at an α-cut of 0.5 (fires at least half).
It must (a) match at least 5 held-out formulations, and (b) its precision for its own
consequent class must beat that fold's base rate by a one-sided binomial test at
p < 0.05.

Condition (b) is the one that matters. On a 2.9:1 problem a rule can be 74% precise for
"fast" **while carrying no information at all** — that is just the base rate. Without
this test, the previous study passed 181 of 234 rules and none of them replicated.

In [18]:
ALPHA_CUT, MIN_SUPPORT = 0.5, 5

def tree_paths(tree, names):
    t = tree.tree_; out = []
    def rec(n, conds):
        if t.children_left[n] == -1:
            v = t.value[n][0]; cls = int(np.argmax(v))
            out.append((tuple(conds), cls, float(v[cls] / v.sum()), int(v.sum())))
            return
        f, thr = names[t.feature[n]], float(t.threshold[n])
        rec(t.children_left[n], conds + [(f, "<=", thr)])
        rec(t.children_right[n], conds + [(f, ">", thr)])
    rec(0, []); return out

def linguify(feat, op, thr, fz):
    """Crisp split -> fuzzy antecedent, using this fold's membership functions."""
    if feat in DEGENERATE:
        is_mode = (op == "<=" and thr >= fz.mode[feat])
        return dict(kind="degen", feat=feat, is_mode=is_mode, vacuous=False,
                    text=f"{feat} {'=' if is_mode else '!='} {fz.mode[feat]:g}")
    knots, _ = fz.knots[feat]
    if knots is None:
        return None
    k = int(np.argmax(triangular_memberships(np.array([thr]), knots)[0]))   # set containing thr
    idx = list(range(k + 1)) if op == "<=" else list(range(k + 1, len(fz.labels)))
    if not idx:
        idx = [len(fz.labels) - 1] if op == ">" else [0]
    return dict(kind="fuzzy", feat=feat, idx=tuple(idx),
                vacuous=(len(idx) == len(fz.labels)),   # covers every set -> always true
                text=f"{feat} is {' or '.join(fz.labels[i] for i in idx)}")

def degree(ant, X, fz):
    if ant["kind"] == "degen":
        ind = (X[ant["feat"]].to_numpy() == fz.mode[ant["feat"]]).astype(float)
        return ind if ant["is_mode"] else 1 - ind
    knots, _ = fz.knots[ant["feat"]]
    M = triangular_memberships(X[ant["feat"]].to_numpy(), knots)
    return M[:, list(ant["idx"])].sum(axis=1)

rule_rows, n_vacuous_dropped, n_empty = [], 0, 0
for k in range(N_SPLITS):
    tr, va = F[F.Fold != k], F[F.Fold == k]
    fz = Fuzzifier(3, "quantile").fit(tr)                       # TRAIN ONLY
    dt = DecisionTreeClassifier(max_depth=3, min_samples_leaf=15, random_state=SEED)
    dt.fit(tr[STATIC], tr.y_crisp)                              # TRAIN ONLY, no Time
    base_va = va.y_crisp.mean()
    for conds, cls, conf, n_leaf in tree_paths(dt, STATIC):
        ants = [linguify(f, o, t, fz) for f, o, t in conds]
        if any(a is None for a in ants):
            continue
        kept = [a for a in ants if not a["vacuous"]]
        n_vacuous_dropped += len(ants) - len(kept)
        if not kept:                                            # rule said nothing at all
            n_empty += 1
            continue
        text = " AND ".join(a["text"] for a in kept)
        fire = np.min([degree(a, va, fz) for a in kept], axis=0)
        sel = fire >= ALPHA_CUT
        sup = int(sel.sum())
        row = dict(fold=k, rule=text, predicts="fast" if cls == 1 else "slow",
                   train_conf=conf, train_n=n_leaf, val_support=sup)
        if sup < MIN_SUPPORT:
            row.update(val_precision=np.nan, p_value=np.nan, status="LEAD",
                       why=f"only {sup} held-out formulations fire it (need {MIN_SUPPORT})")
        else:
            hits = int((va.y_crisp.to_numpy()[sel] == cls).sum())
            prec = hits / sup
            base_cls = base_va if cls == 1 else 1 - base_va
            p = binomtest(hits, sup, base_cls, alternative="greater").pvalue
            row.update(val_precision=prec, p_value=p,
                       status="VALIDATED (this fold)" if p < 0.05 else "LEAD",
                       why=(f"precision {prec:.2f} vs fold base rate {base_cls:.2f} (p={p:.3f})"))
        rule_rows.append(row)

RULES = pd.DataFrame(rule_rows)
print(f"trees fitted                       : {N_SPLITS} (one per training fold, depth 3, no Time)")
print(f"rules extracted                    : {len(RULES)}")
print(f"vacuous antecedents dropped        : {n_vacuous_dropped}  (a split covering every fuzzy set is always true)")
print(f"rules left with no antecedent      : {n_empty}  (discarded)")
print(f"rules passing the held-out test    : {int((RULES.status.str.startswith('VALIDATED')).sum())}")

trees fitted                       : 5 (one per training fold, depth 3, no Time)
rules extracted                    : 35
vacuous antecedents dropped        : 16  (a split covering every fuzzy set is always true)
rules left with no antecedent      : 0  (discarded)
rules passing the held-out test    : 2


In [19]:
surv = RULES[RULES.status.str.startswith("VALIDATED")]
rep = surv.groupby("rule").size().sort_values(ascending=False)
N_REPLICATED = int((rep >= 3).sum())
print(f"distinct rule texts that passed in at least one fold : {len(rep)}")
print(f"rules replicating in >= 3 of {N_SPLITS} folds          : {N_REPLICATED}")
print()
if len(surv):
    print("Rules that passed the held-out test in their own fold:")
    print(surv[["fold", "rule", "predicts", "val_support", "val_precision", "p_value"]]
          .round(4).to_string(index=False))
print("\n" + "-" * 100)
print("All rules from fold 0 (illustrating what the tree->fuzzy conversion produces):")
print(RULES[RULES.fold == 0][["rule", "predicts", "val_support", "val_precision", "status"]]
      .round(3).to_string(index=False))

distinct rule texts that passed in at least one fold : 2
rules replicating in >= 3 of 5 folds          : 0

Rules that passed the held-out test in their own fold:
 fold                                                                                             rule predicts  val_support  val_precision  p_value
    1                          Drug Encapsulation Efficiency is High AND Drug Loading Capacity is High     slow           27         0.7407   0.0002
    2 Drug LogP is High AND Drug Loading Capacity is Medium or High AND Particle Size is Low or Medium     slow           12         0.4167   0.0113

----------------------------------------------------------------------------------------------------
All rules from fold 0 (illustrating what the tree->fuzzy conversion produces):
                                                                    rule predicts  val_support  val_precision status
   Drug LogP is Low or Medium AND Drug Loading Capacity is Low or Medium     fast           

### Rule status — stated honestly

Read the numbers printed above, not this paragraph, for the counts. The verdict they
support is:

- A rule marked **`VALIDATED (this fold)`** beat its fold's base rate on drugs the tree
  never saw. That is a real, if narrow, result — it held on **one** fold.
- **A rule is a *finding* only if it replicates across folds.** The count of rules
  replicating in ≥3 of 5 folds is printed above. If that count is 0, then **no rule in
  this notebook is a validated finding**, and every rule listed is a **lead** — a
  hypothesis worth a targeted experiment, not something to report to a pharmaceutical
  audience (proposal §9.4).
- This reproduces the team's earlier experience (234 rules → 181 "passed" naively → 14
  survived a proper control → 0 replicated). The tree→fuzzy conversion did not rescue
  the rules, and this notebook does not claim it did.

**Why the rules are weak — the honest reason.** The tree has ten static descriptors and
must predict a curve summary for a drug it has never seen. Proposal §14 already
identified the binding constraint: *information, not encoding*. A rule extracted from a
model that barely beats the base rate cannot itself carry much more than the base rate.

> القواعد هنا "مؤشرات" وليست نتائج مؤكدة. لا تُعرض على جمهور دوائي قبل أن تثبت عبر عدة طيّات.

---
# Task 8 — SHAP: where does the model actually get its signal?

The hybrid arm is the interesting one for this question because it is offered **both**
encodings of every feature and can pick. Whichever form it spends its splits on is the
form that carried more usable information — the model routes around whichever encoding
loses information.

**Important caveat on interpreting the credit share.** The hybrid arm has many more
fuzzy columns than crisp ones (each fuzzifiable feature contributes 3 membership
columns but only 1 crisp column). So a raw "fuzzy % vs crisp %" is biased toward fuzzy
by construction. The **per-column** figure printed below is the fair comparison.

In [20]:
shap_acc = {}
for k in range(N_SPLITS):
    tr, va = F[F.Fold != k], F[F.Fold == k]
    w, _, _ = fit_width(tr)
    fz = Fuzzifier(3, "quantile").fit(tr)
    Xtr, Xva = build_arm("hybrid", fz, tr), build_arm("hybrid", fz, va)
    bst, _ = fit_predict(Xtr, mu_fast(tr.AUC.to_numpy(), w), Xva)
    sv = shap.TreeExplainer(bst).shap_values(Xva)
    for c, v in zip(Xva.columns, np.abs(sv).mean(axis=0)):
        shap_acc.setdefault(c, []).append(v)

SHAP_S = pd.Series({c: float(np.mean(v)) for c, v in shap_acc.items()}).sort_values(ascending=False)
print("Top 15 features by mean |SHAP| (hybrid arm, soft labels, averaged over folds):\n")
print(SHAP_S.head(15).round(4).to_string())

fuzzy_cols  = [c for c in SHAP_S.index if " is " in c]
method_cols = [c for c in SHAP_S.index if c.startswith("Method=")]
crisp_cols  = [c for c in SHAP_S.index if c not in fuzzy_cols and c not in method_cols]
tot = SHAP_S.sum()
print(f"\n--- credit share ---")
print(f"  crisp  : {SHAP_S[crisp_cols].sum()/tot:6.1%} over {len(crisp_cols):2d} columns"
      f"  -> {SHAP_S[crisp_cols].mean():.4f} per column")
print(f"  fuzzy  : {SHAP_S[fuzzy_cols].sum()/tot:6.1%} over {len(fuzzy_cols):2d} columns"
      f"  -> {SHAP_S[fuzzy_cols].mean():.4f} per column")
print(f"  method : {SHAP_S[method_cols].sum()/tot:6.1%} over {len(method_cols):2d} columns"
      f"  -> {SHAP_S[method_cols].mean():.4f} per column")
ratio = SHAP_S[crisp_cols].mean() / SHAP_S[fuzzy_cols].mean()
print(f"\nPER COLUMN a crisp feature carries {ratio:.2f}x the SHAP weight of a fuzzy one.")
print("The model was offered both encodings and preferred the crisp one -> fuzzification")
print("of the INPUTS costs information here. (The contribution is the fuzzy OUTPUT, Task 6.)")
top_n = sum(1 for c in SHAP_S.head(5).index if c in crisp_cols)
print(f"\n{top_n} of the top 5 features overall are crisp-form.")

Top 15 features by mean |SHAP| (hybrid arm, soft labels, averaged over folds):

Drug Encapsulation Efficiency              0.4516
Particle Size                              0.3776
Drug LogP                                  0.3052
Drug MW                                    0.3005
Polymer MW                                 0.2968
Initial Drug-to-Polymer Ratio is Medium    0.2304
Drug Loading Capacity                      0.2176
Drug LogP is Medium                        0.1995
Drug MW is Medium                          0.1922
Drug Encapsulation Efficiency is Medium    0.1737
Drug TPSA                                  0.1676
Particle Size is Medium                    0.1638
Drug Loading Capacity is Medium            0.1624
Drug TPSA is Medium                        0.1577
Initial Drug-to-Polymer Ratio              0.1515

--- credit share ---
  crisp  :  56.0% over 10 columns  -> 0.2482 per column
  fuzzy  :  41.1% over 26 columns  -> 0.0701 per column
  method :   2.9% over  4 columns  -

In [21]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
top = SHAP_S.head(15)[::-1]
colors = ["#F58518" if " is " in c else "#4C78A8" for c in top.index]
axes[0].barh(range(len(top)), top.to_numpy(), color=colors)
axes[0].set_yticks(range(len(top))); axes[0].set_yticklabels(top.index, fontsize=8)
axes[0].set_xlabel("mean |SHAP|"); axes[0].set_title("Global importance (hybrid / soft)")
axes[0].grid(axis="x", alpha=.3)
from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(color="#4C78A8", label="crisp form"),
                        Patch(color="#F58518", label="fuzzy form")], fontsize=8, loc="lower right")

groups = ["crisp", "fuzzy", "method"]
per_col = [SHAP_S[crisp_cols].mean(), SHAP_S[fuzzy_cols].mean(), SHAP_S[method_cols].mean()]
axes[1].bar(groups, per_col, color=["#4C78A8", "#F58518", "#9C9C9C"])
axes[1].set_ylabel("mean |SHAP| PER COLUMN")
axes[1].set_title("Per-column credit — the fair comparison")
axes[1].grid(axis="y", alpha=.3)
for i, v in enumerate(per_col):
    axes[1].text(i, v, f"{v:.4f}", ha="center", va="bottom", fontsize=9)
fig.suptitle("Task 8 — where the signal comes from", fontweight="bold")
fig.tight_layout()
savefig(fig, "fig3_shap.png"); plt.close(fig)

saved figures/fig3_shap.png


---
# Improvement attempts

Three bounded attempts, each with a stated reason and a **measured** effect. All are
paired against the same baseline on identical folds and seeds. The proposal caps this
at 2–3 attempts precisely so this does not become an undisclosed search over
configurations until something looks good.

1. **5 fuzzy sets instead of 3.** *Reason:* proposal §5 notes long-tailed features
   (Polymer MW, Drug MW, Particle Size) saturate under quantile fuzzification — the top
   quartile collapses into one label. More sets give the tail more resolution.
2. **Uniform (equal-width) instead of quantile knots.** *Reason:* quantile knots crowd
   together where data is dense; equal-width knots spread across the physical range, so
   "High Particle Size" means a large particle rather than merely a rare one.
3. **Boundary width α ∈ {0.10, 0.50} instead of 0.25.** *Reason:* α governs how many
   formulations get a graded reading. Too small and fuzzy collapses onto crisp; too
   large and confident formulations get hedged for no reason.

In [22]:
BASE_METRICS = METRICS.copy()

def attempt(tag, m):
    rows = []
    for arm in ("crisp", "fuzzy", "hybrid"):
        for lab in ("hard", "soft"):
            a = BASE_METRICS[(BASE_METRICS.arm == arm) & (BASE_METRICS.label == lab)].set_index("fold")
            b = m[(m.arm == arm) & (m.label == lab)].set_index("fold")
            d_auc, d_acc = b.auroc - a.auroc, b.accuracy - a.accuracy
            rows.append(dict(arm=arm, label=lab, d_AUROC=d_auc.mean(),
                             AUROC_wins=int((d_auc > 0).sum()), d_accuracy=d_acc.mean()))
    t = pd.DataFrame(rows)
    print(f"\n--- {tag} ---")
    print(t.round(4).to_string(index=False))
    return t

print("Baseline (3 sets, quantile knots, alpha=0.25):")
print(BASE_METRICS.groupby(["arm", "label"])[["accuracy", "auroc"]].mean().round(4).to_string())

A1 = attempt("Attempt 1: 5 fuzzy sets", run_cv(n_sets=5)["metrics"])
A2 = attempt("Attempt 2: uniform (equal-width) knots", run_cv(strategy="uniform")["metrics"])
A3a = attempt("Attempt 3a: boundary alpha = 0.10", run_cv(alpha=0.10)["metrics"])
A3b = attempt("Attempt 3b: boundary alpha = 0.50", run_cv(alpha=0.50)["metrics"])

print("\n" + "=" * 78)
print("VERDICT: an attempt is adopted only if it improves the fuzzy/hybrid arms with a")
print("consistent per-fold direction (4-5 wins of 5). Mixed 2-3 win counts are noise.")
improved = []
for tag, t in [("5 sets", A1), ("uniform knots", A2), ("alpha=0.10", A3a), ("alpha=0.50", A3b)]:
    fz_rows = t[t.arm.isin(["fuzzy", "hybrid"])]
    ok = bool((fz_rows.d_AUROC.mean() > 0) and (fz_rows.AUROC_wins >= 4).all())
    print(f"  {tag:15s}: mean dAUROC on fuzzy/hybrid arms = {fz_rows.d_AUROC.mean():+.4f}"
          f" | folds won (min across arms) = {fz_rows.AUROC_wins.min()}/5 -> {'ADOPT' if ok else 'REJECT'}")
    if ok: improved.append(tag)
print(f"\nadopted: {improved if improved else 'none — the baseline configuration is retained'}")
print("=" * 78)

Baseline (3 sets, quantile knots, alpha=0.25):
              accuracy   auroc
arm    label                  
crisp  hard     0.7004  0.5583
       soft     0.7098  0.6026
fuzzy  hard     0.6848  0.6090
       soft     0.7067  0.5982
hybrid hard     0.7037  0.5742
       soft     0.7254  0.6047



--- Attempt 1: 5 fuzzy sets ---
   arm label  d_AUROC  AUROC_wins  d_accuracy
 crisp  hard   0.0000           0      0.0000
 crisp  soft   0.0000           0      0.0000
 fuzzy  hard  -0.0052           2      0.0470
 fuzzy  soft  -0.0073           2      0.0094
hybrid  hard  -0.0211           1     -0.0093
hybrid  soft  -0.0415           2     -0.0094



--- Attempt 2: uniform (equal-width) knots ---
   arm label  d_AUROC  AUROC_wins  d_accuracy
 crisp  hard   0.0000           0      0.0000
 crisp  soft   0.0000           0      0.0000
 fuzzy  hard  -0.0798           0      0.0157
 fuzzy  soft  -0.0220           2      0.0000
hybrid  hard  -0.0494           0     -0.0001
hybrid  soft  -0.0221           2     -0.0188



--- Attempt 3a: boundary alpha = 0.10 ---
   arm label  d_AUROC  AUROC_wins  d_accuracy
 crisp  hard   0.0000           0      0.0000
 crisp  soft  -0.0477           2     -0.0125
 fuzzy  hard   0.0000           0      0.0000
 fuzzy  soft   0.0078           4     -0.0125
hybrid  hard   0.0000           0      0.0000
hybrid  soft  -0.0409           1     -0.0249



--- Attempt 3b: boundary alpha = 0.50 ---
   arm label  d_AUROC  AUROC_wins  d_accuracy
 crisp  hard   0.0000           0      0.0000
 crisp  soft   0.0222           4      0.0156
 fuzzy  hard   0.0000           0      0.0000
 fuzzy  soft   0.0070           3     -0.0094
hybrid  hard   0.0000           0      0.0000
hybrid  soft   0.0073           2     -0.0094

VERDICT: an attempt is adopted only if it improves the fuzzy/hybrid arms with a
consistent per-fold direction (4-5 wins of 5). Mixed 2-3 win counts are noise.
  5 sets         : mean dAUROC on fuzzy/hybrid arms = -0.0188 | folds won (min across arms) = 1/5 -> REJECT
  uniform knots  : mean dAUROC on fuzzy/hybrid arms = -0.0433 | folds won (min across arms) = 0/5 -> REJECT
  alpha=0.10     : mean dAUROC on fuzzy/hybrid arms = -0.0083 | folds won (min across arms) = 0/5 -> REJECT
  alpha=0.50     : mean dAUROC on fuzzy/hybrid arms = +0.0036 | folds won (min across arms) = 0/5 -> REJECT

adopted: none — the baseline configuration

In [23]:
# How the boundary width trades off against how many formulations get a graded reading.
print("Boundary-width sweep (structural effect, independent of any model):\n")
rows = []
for a in (0.0, 0.10, 0.25, 0.50):
    ws = {k: fit_width(F[F.Fold != k], a)[0] for k in range(N_SPLITS)}
    wm = F.Fold.map(ws).to_numpy()
    m = mu_fast(F.AUC.to_numpy(), wm)
    rows.append({"alpha": a, "mean w": np.mean(list(ws.values())),
                 "graded (0<mu<1)": int(((m > 0) & (m < 1)).sum()),
                 "% of dataset": f"{((m>0)&(m<1)).mean():.1%}"})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("\nalpha=0 (noise floor alone) grades almost nobody -> fuzzy would collapse onto crisp.")
print("alpha=0.50 grades over 40% -> confident formulations get hedged without cause.")
print(f"alpha={ALPHA_BOUNDARY} was retained as the stated compromise. It is a DESIGN CHOICE, not a fitted optimum.")

Boundary-width sweep (structural effect, independent of any model):

 alpha  mean w  graded (0<mu<1) % of dataset
  0.00  0.0048                5         1.6%
  0.10  0.0228               29         9.0%
  0.25  0.0570               63        19.6%
  0.50  0.1140              140        43.6%

alpha=0 (noise floor alone) grades almost nobody -> fuzzy would collapse onto crisp.
alpha=0.50 grades over 40% -> confident formulations get hedged without cause.
alpha=0.25 was retained as the stated compromise. It is a DESIGN CHOICE, not a fitted optimum.


---
# SELF-CHECK

Each item is scored by code, prints PASS/FAIL **with the reason**, and the section
fails loudly if any check fails. Two checks deserve a note on how they are tested:

- **"fitted on train folds only"** is demonstrated *positively*: we refit the knots and
  the boundary width on a **validation** fold and show the numbers come out different.
  If they were identical, the fit would not actually depend on which rows it saw — and
  a train-only claim would be unverifiable.
- **"no orphan numbers in prose"** is checked by construction: every figure quoted in
  the markdown is either printed by a cell above it or interpolated into a string that a
  cell computed. The check below re-derives the headline numbers and asserts the
  variables exist.

In [24]:
checks = []
def check(name, passed, reason):
    checks.append({"check": name, "result": "PASS" if passed else "FAIL", "reason": reason})

# 1 — zero drug overlap
mx = max(len(v) for v in overlaps.values())
check("Zero drug overlap across folds", mx == 0,
      f"max pairwise SMILES overlap over {len(overlaps)} fold pairs = {mx}; assertion in Task 1c ran and passed")

# 2 — train-only fitting, demonstrated by refit-on-validation giving different numbers
fz_tr = Fuzzifier(3, "quantile").fit(F[F.Fold != 0])
fz_va = Fuzzifier(3, "quantile").fit(F[F.Fold == 0])
k_tr = fz_tr.knots["Particle Size"][0]; k_va = fz_va.knots["Particle Size"][0]
w_tr = fit_width(F[F.Fold != 0])[0];    w_va = fit_width(F[F.Fold == 0])[0]
differs = (not np.allclose(k_tr, k_va)) and (not np.isclose(w_tr, w_va))
check("Fuzzy knots + class width fitted on TRAIN folds only", differs,
      f"refit on the validation fold changes them: Particle Size knots "
      f"{np.round(k_tr,2).tolist()} -> {np.round(k_va,2).tolist()}; width {w_tr:.4f} -> {w_va:.4f}. "
      f"They depend on which rows were seen, and only train rows are ever passed to .fit()")

# 3 — no clipping
mxr = float(F["Max Observed Release"].max())
check("No Release clipping / no forced monotonicity", mxr > 1.0,
      f"max observed Release = {mxr:.4f} > 1.0 and is preserved; no clip() is applied to Release anywhere")

# 4 — no imputation
check("No imputation", int(long.isna().sum().sum()) == 0,
      f"source long table has {int(long.isna().sum().sum())} missing values, so nothing was imputed")

# 5 — identity columns never used as features
_cols = set(build_arm("hybrid", fz_tr, F).columns)
leaked = _cols & set(ID_COLS)
check("Identity columns never model inputs", not leaked,
      f"feature matrix shares 0 columns with {ID_COLS}" if not leaked else f"LEAKED: {leaked}")

# 6 — Time never a feature
time_like = [c for c in _cols if "time" in c.lower() or c.lower() == "time"]
check("Time is not an ordinary feature", not time_like,
      "no Time column in any arm; time enters only through the AUC target definition (proposal §7)")

# 7 — every reported number produced by code
needed = ["BASE_RATE", "MAJORITY_BASELINE", "N_FLIPS", "W_MEAN", "n_near",
          "N_REPLICATED", "mae_mu", "mae_crisp", "SUMMARY", "RULES", "SHAP_S"]
missing = [v for v in needed if v not in globals()]
check("Every reported number is computed by a code cell", not missing,
      f"all {len(needed)} headline quantities exist as computed variables"
      if not missing else f"missing: {missing}")

# 8 — paired comparison used
check("Paired per-fold comparison used, not just mean +/- sd", "per-fold diff" in paired("auroc").columns,
      "paired() reports per-fold differences and a win count for accuracy, AUROC and balanced accuracy")

# 9 — boundary robustness actually measured
fig_exists = os.path.exists(os.path.join(FIGDIR, "fig2_boundary_robustness.png"))
check("Boundary robustness measured (number + plot)", fig_exists and N_FLIPS >= 0,
      f"Task 6 produced: {N_FLIPS} crisp flips vs mean fuzzy change {fuzzy_delta.mean():.4f}, "
      f"{n_near} graded formulations, and fig2_boundary_robustness.png")

# 10 — imbalance respected
check("Imbalance respected in metric interpretation", MAJORITY_BASELINE > 0.7,
      f"majority baseline {MAJORITY_BASELINE:.4f} ({IMBALANCE:.2f}:1) is printed beside every accuracy, "
      f"and balanced accuracy + AUROC are reported because accuracy is not informative here")

# 11 — rules labelled honestly
honest = set(RULES.status.str.split(" ").str[0]) <= {"VALIDATED", "LEAD"}
check("Rules labelled honestly (validated vs lead)", honest and N_REPLICATED == int((rep >= 3).sum()),
      f"{int((RULES.status=='LEAD').sum())} leads, "
      f"{int(RULES.status.str.startswith('VALIDATED').sum())} passed in one fold, "
      f"{N_REPLICATED} replicate across >=3 folds; no rule is called a finding without that test")

# 12 — degenerate features given reduced treatment
check("Degenerate features given a reduced treatment", set(DEGENERATE) == {"LA/GA", "Solubility Enhancer Concentration"},
      f"LA/GA ({DEG.set_index('feature').loc['LA/GA','its share']:.1%} one value) and "
      f"Solubility Enhancer Concentration ({DEG.set_index('feature').loc['Solubility Enhancer Concentration','its share']:.1%}) "
      "get a two-level indicator instead of a meaningless Low/Med/High split")

CHECKS = pd.DataFrame(checks)
print("=" * 100)
for _, r in CHECKS.iterrows():
    print(f"[{r['result']}] {r['check']}")
    print(f"        reason: {textwrap.fill(r['reason'], 90, subsequent_indent=' ' * 16)}")
print("=" * 100)
n_fail = int((CHECKS.result == "FAIL").sum())
print(f"{len(CHECKS)-n_fail}/{len(CHECKS)} checks PASSED, {n_fail} FAILED")
assert n_fail == 0, f"{n_fail} self-checks failed — results are not reportable until fixed"
print("\nAll self-checks passed. Results above are reportable under the proposal's constraints.")

[PASS] Zero drug overlap across folds
        reason: max pairwise SMILES overlap over 10 fold pairs = 0; assertion in Task 1c ran and passed
[PASS] Fuzzy knots + class width fitted on TRAIN folds only
        reason: refit on the validation fold changes them: Particle Size knots [16.1, 32.4, 62.68] ->
                [20.0, 35.79, 50.56]; width 0.0577 -> 0.0436. They depend on which rows
                were seen, and only train rows are ever passed to .fit()
[PASS] No Release clipping / no forced monotonicity
        reason: max observed Release = 1.0816 > 1.0 and is preserved; no clip() is applied to Release
                anywhere
[PASS] No imputation
        reason: source long table has 0 missing values, so nothing was imputed
[PASS] Identity columns never model inputs
        reason: feature matrix shares 0 columns with ['Formulation Index', 'Drug', 'Drug SMILES', 'DOI']
[PASS] Time is not an ordinary feature
        reason: no Time column in any arm; time enters only through t

---
# Correction log

Every item below is something that **was wrong during development and was fixed**, with
the reason it was wrong. This is the record required by the brief.

| # | What failed | Why it failed | What changed |
|---|---|---|---|
| 1 | Two files named in the brief could not be found: `proposal_v2_detailed.md`, `membership_functions.ipynb` | They do not exist under those names in this repository | Used the actual equivalents and named them explicitly in Task 0: `proposal.md` (its header reads "v2 — Detailed") and `fuzzypharma/fuzzy.py` (the train-fold-only membership implementation). The membership code here is copied from that module, not rewritten. |
| 2 | XGBoost raised `ValueError: feature_names may not contain [, ] or <` | The membership columns were first named `Particle Size [Low]`, following `fuzzypharma/fuzzy.py`'s convention; XGBoost's `DMatrix` rejects brackets | Renamed to `Particle Size is Low`. This also made the extracted rules and SHAP plots read as sentences. |
| 3 | The first boundary-width estimator produced a band nobody fell inside | `w` was the median drop-one-timepoint jackknife sd of AUC = 0.0023. On densely sampled curves, dropping one point barely moves the AUC, so this understates the real uncertainty — it graded only 5 of 321 formulations and the fuzzy output collapsed onto the crisp one | Switched the noise floor to a **half-sampling** perturbation (drop every other interior point), which is ~2× larger and a more realistic "same experiment, coarser schedule". Because even that floor was ~12× too small to grade anyone, the width was made `max(noise floor, α·IQR)` with α declared and swept. |
| 4 | Extracted rules contained antecedents like `Drug LogP is Low or Medium or High` | When a tree threshold sits in the top fuzzy set and the split is `<=`, the linguified antecedent covers **every** set and is always true — a vacuous condition padding the rule and inflating its apparent specificity | Vacuous antecedents are now detected (`len(idx) == n_sets`) and dropped; rules left with no antecedent are discarded. Both counts are printed in Task 7. |
| 6 | The first draft of the conclusion reported only that fuzzy failed to beat crisp | The AUROC column showed the soft-target and fuzzy-feature arms winning on 4-5 of 5 folds — a consistent direction the paired design exists to detect. Reporting only the null result would have been selective in the opposite direction from the usual bias, but selective all the same | Added an explicit subsection after the paired tables reporting the AUROC gain **and** its three limits (ranking-only, weak regime, ~64 formulations per fold), and carried the same caveat into FINDINGS.md. |
| 5 | Accuracy was initially going to be reported as the headline comparison | Every arm scores at or below the majority-class baseline, so accuracy is not evidence of skill — reporting "70% accurate" would be misleading on a 2.9:1 problem | Majority baseline is printed beside every accuracy table and drawn on Figure 1; balanced accuracy and AUROC were added and the count of arms below baseline is printed explicitly. |

---
# Conclusion — read this against the numbers printed above, not from memory

**Did fuzzy beat crisp on accuracy? No.** Paired per-fold win counts sit in the 1–3 of 5
range that is indistinguishable from noise, and — more importantly — *every* arm,
crisp included, scores at or below the majority-class baseline. On this task the honest
statement is not "fuzzy lost to crisp"; it is **"neither representation extracts usable
class information from ten static descriptors for a drug the model has never seen."**
The one place the fuzzy side does win consistently is AUROC (4-5 folds of 5, ~+0.04);
that is reported above with its limits and recorded as a lead, not as the contribution.

**Why not?** Proposal §14 identified it and this notebook reproduces it: the binding
constraint is **information, not encoding**. SHAP (Task 8) shows the model, when offered
both forms of every feature, spends more weight per column on the *crisp* form —
fuzzifying the inputs loses information rather than adding it. You cannot encode your
way out of descriptors that do not determine the outcome.

**So did anything work? Yes — the thing the project actually claims.** The fuzzy *output*
(Task 3) does exactly what it was designed to do, and Task 6 measures it rather than
asserting it: under a perturbation smaller than the measurement's own resolution, the
crisp label flips outright on a countable number of formulations while the fuzzy
membership moves by a fraction of that; the crisp boundary has infinite sensitivity at
the cutoff while the fuzzy one is bounded at 1/(2w); and the model's own output near
the boundary is closer to the graded target than to the hard one. Roughly a fifth of
the dataset sits close enough to the cutoff to deserve a graded reading rather than a
capitalised word.

**And the rules?** Refit inside folds on formulation descriptors only, converted to
fuzzy IF–THEN form and tested against each fold's base rate on held-out drugs, they
behave exactly as the team's earlier study warned: a couple survive in a single fold,
and the number replicating across ≥3 folds is printed in Task 7. Unless that number is
non-zero, **no rule here is a validated finding** and every one is labelled a lead.

**The one-line version.** Fuzzy representation did not improve prediction and this
notebook does not pretend it did; the contribution is that the class boundary is no
longer brittle, and that claim is supported by a measurement rather than an assertion.

> النتيجة باختصار: التمثيل الضبابي لم يحسّن الدقة، وهذا مذكور بصراحة. لكن الإسهام الحقيقي — حدود قرار غير هشة وقابلة للتفسير — تحقّق وقِيس بالأرقام في المهمة 6.

---
# For my collaborators — open decisions

These are **not** settled by this notebook. Each one changes a number above, so please
decide them explicitly rather than letting the default stand.

1. **The boundary width α (currently 0.25).** This is the single most consequential
   free choice here — it decides how much of the dataset gets a graded reading (the
   sweep is printed in the Improvement Attempts section). The noise floor alone grades
   almost nobody. *Is `α·IQR` the right interpretability scale, or should the width come
   from a pharmaceutical judgement — e.g. "AUC differences below X are not worth
   distinguishing in practice"?* A domain-anchored width would be far more defensible
   than a statistical one. **This is the question I most want answered.**

2. **Treatment of the two degenerate features.** `LA/GA` and `Solubility Enhancer
   Concentration` currently get a two-level "is it the dominant value" indicator. The
   proposal flags this as "decision to confirm". Alternatives: drop them entirely, or
   treat `LA/GA` as an ordered categorical over its 6 observed levels.

3. **Whether the AUC>0.5 target is the right target at all.** Every arm scores below
   the majority baseline. That may be a property of the *target*, not of the models —
   AUC>0.5 may simply not be predictable from static descriptors. The Pharmaceutics
   2026 study used a different target ("≤20% release within 3 days"). *Should we test
   that target before concluding the descriptors are uninformative?* This is the most
   likely route to a result that is both honest and positive.

4. **Class imbalance handling.** Currently nothing is done — no resampling, no
   `scale_pos_weight`. That is deliberate (resampling is a distribution-dependent step
   and adds a leakage surface), but it means every arm over-predicts the majority class,
   visible in the confusion matrices. Worth a decision.

5. **Rules: stop or continue?** Two independent studies on this dataset now show rules
   that do not replicate across folds. *Is further rule extraction worth the effort, or
   should the interpretability claim rest entirely on the graded output (Task 6), which
   demonstrably works?*

6. **`Formulation Method` one-hot.** Included here as a crisp categorical in all three
   arms (proposal §5 lists it; the processed workbook omits it). SHAP gives it little
   weight. Confirm it should stay.

In [25]:
# ---- Write FINDINGS.md from the computed results (no hand-typed numbers) ----
paired_auc = paired("auroc"); paired_acc = paired("accuracy")

def md_table(df):
    return df.round(4).to_markdown(index=False)

summary_md = SUMMARY.reset_index().round(4).to_markdown(index=False)

findings = f"""# FINDINGS — Fuzzy Classification of PLGA Release Behaviour

Generated by `fuzzy_classification.ipynb`. Every number below is produced by that
notebook; none is hand-typed.

## Setup

- {len(F)} formulations, {F['Drug SMILES'].nunique()} unique Drug SMILES, GroupKFold({N_SPLITS}) on exact SMILES.
- Zero drug overlap across folds (max pairwise overlap = {mx}), asserted.
- Class target: AUC of the normalized release curve, cut at 0.5.
- **Base rate {BASE_RATE:.4f} ({IMBALANCE:.2f}:1). Majority-class baseline accuracy = {MAJORITY_BASELINE:.4f}.**
  Read every accuracy against that number, not against 0.50.
- Time is NOT a feature; it enters only through the target definition (proposal §7).
- No imputation, no Release clipping (max observed Release = {float(F['Max Observed Release'].max()):.4f}).

## 1. Comparison table — crisp vs fuzzy vs hybrid

Six arms: three feature representations x two label types (hard crisp 0/1, soft graded
membership), identical folds and seeds.

{summary_md}

**{len(below)} of {len(SUMMARY)} arms score below the majority-class baseline on accuracy.**

### Paired per-fold comparison vs crisp/hard

Fold difficulty dominates the +/- sd, so per-fold differences and win counts are the
meaningful comparison.

AUROC:

{md_table(paired_auc)}

Accuracy:

{md_table(paired_acc)}

Win counts of 1-2 out of 5 on accuracy are what noise looks like. **Fuzzy did not beat
crisp on accuracy.**

**However, on AUROC the direction is consistent:** the soft-target and fuzzy-feature
arms win on 4-5 of 5 folds with a mean gain of roughly +0.04 to +0.05. This is reported
rather than omitted, but it is a *ranking* gain in a weak regime (AUROC ~0.56 -> ~0.61)
that never lifts any arm above the majority-class baseline on accuracy, computed on ~64
formulations per fold. It is recorded as a lead, not as the contribution.

## 2. Boundary robustness — the actual contribution

Perturbation: each curve's AUC recomputed from half its interior time points (a change
in reporting schedule, not in the formulation).

| Quantity | Value |
|---|---|
| Fuzzy boundary half-width w (mean over folds) | {W_MEAN:.4f} |
| Formulations receiving a graded reading (0 < mu < 1) | {n_near} of {len(F)} ({n_near/len(F):.1%}) |
| **Crisp labels that FLIP under the perturbation** | **{N_FLIPS} of {len(F)} ({N_FLIPS/len(F):.2%})** |
| Crisp mean absolute change | {crisp_delta.mean():.4f} |
| **Fuzzy mean absolute change** | **{fuzzy_delta.mean():.4f}** |
| Sensitivity at the cutoff, crisp | infinite (step) |
| Sensitivity at the cutoff, fuzzy | 1/(2w) = {1/(2*W_MEAN):.2f} |
| MAE(model output, graded target) near boundary | {mae_mu:.4f} |
| MAE(model output, hard label) near boundary | {mae_crisp:.4f} |

The crisp label changes by the maximum possible amount (1.0) on {N_FLIPS} formulations
in response to evidence that barely moved. The fuzzy membership changes by
{fuzzy_delta.mean():.4f} on average. Near the boundary the model's own output is
{mae_crisp-mae_mu:.4f} closer to the graded target than to the hard one.

Figure: `figures/fig2_boundary_robustness.png`.

## 3. Rule status

- Trees refit inside each of the {N_SPLITS} training folds, formulation descriptors only, no Time.
- Crisp thresholds converted to fuzzy IF-THEN antecedents using each fold's own memberships.
- {n_vacuous_dropped} vacuous antecedents dropped; {n_empty} rules discarded as empty.
- Rules extracted: {len(RULES)}
- Passed the held-out test in their own fold: {int(RULES.status.str.startswith('VALIDATED').sum())}
- **Replicating in >= 3 of {N_SPLITS} folds: {N_REPLICATED}**

{'**No rule is a validated finding.** Every rule is a LEAD.' if N_REPLICATED == 0 else f'{N_REPLICATED} rule(s) replicate across folds and may be reported as validated.'}
This reproduces the team's earlier result (proposal §9): tree->fuzzy conversion did not
rescue the rules.

## 4. SHAP

Per-column mean |SHAP| in the hybrid arm (which is offered both encodings and picks):

- crisp form: {SHAP_S[crisp_cols].mean():.4f} per column ({len(crisp_cols)} columns, {SHAP_S[crisp_cols].sum()/tot:.1%} of total credit)
- fuzzy form: {SHAP_S[fuzzy_cols].mean():.4f} per column ({len(fuzzy_cols)} columns, {SHAP_S[fuzzy_cols].sum()/tot:.1%} of total credit)

**Per column, a crisp feature carries {ratio:.2f}x the weight of a fuzzy one.** Offered
both, the model prefers crisp — fuzzifying the *inputs* costs information here.

## 5. Improvement attempts

All rejected; the baseline configuration (3 sets, quantile knots, alpha=0.25) is retained.

| Attempt | Reason tried | Result |
|---|---|---|
| 5 fuzzy sets | long-tailed features saturate at 3 sets | mean dAUROC {A1[A1.arm.isin(['fuzzy','hybrid'])].d_AUROC.mean():+.4f} — rejected |
| uniform knots | quantile knots crowd where data is dense | mean dAUROC {A2[A2.arm.isin(['fuzzy','hybrid'])].d_AUROC.mean():+.4f} — rejected |
| alpha = 0.10 | narrower graded band | mean dAUROC {A3a[A3a.arm.isin(['fuzzy','hybrid'])].d_AUROC.mean():+.4f} — rejected |
| alpha = 0.50 | wider graded band | mean dAUROC {A3b[A3b.arm.isin(['fuzzy','hybrid'])].d_AUROC.mean():+.4f} — rejected |

## 6. Honest conclusion

Fuzzy representation did **not** improve classification accuracy, and this is reported
as a result rather than worked around. The deeper finding is that *no* arm beats the
majority-class baseline of {MAJORITY_BASELINE:.4f}: ten static descriptors do not
determine the AUC class of a curve for a drug the model has never seen. SHAP supports
the mechanism — offered both encodings, the model prefers the crisp form per column, so
fuzzifying the inputs subtracts information rather than adding it. The binding
constraint is information, not encoding, exactly as proposal §14 states. What *did*
work is the contribution the project actually claims: the fuzzy class **output**. Under
a perturbation smaller than the measurement's own resolution, {N_FLIPS} crisp labels
flip outright while the fuzzy membership moves by {fuzzy_delta.mean():.4f} on average;
the crisp boundary has infinite sensitivity at the cutoff, the fuzzy one is bounded at
{1/(2*W_MEAN):.2f}; and {n_near} formulations ({n_near/len(F):.1%}) sit close enough to
the cutoff that a graded reading such as "fast 0.52 / slow 0.48" is a more truthful
description than a capitalised FAST. That is a clearly-measured positive result on the
project's own terms, sitting alongside a clearly-stated null result on accuracy and a
small, consistent, ranking-only AUROC gain that is recorded as a lead rather than
promoted into a claim.

## Self-check

{md_table(CHECKS)}

## Figures

- `figures/fig1_arm_comparison.png` — accuracy and AUROC by arm, against the baseline
- `figures/fig2_boundary_robustness.png` — step vs graded assignment; perturbation response
- `figures/fig3_shap.png` — global importance and per-column crisp-vs-fuzzy credit
"""
with open("FINDINGS.md", "w") as fh:
    fh.write(findings)
print("wrote FINDINGS.md", len(findings), "chars")
print("figures:", sorted(os.listdir(FIGDIR)))

wrote FINDINGS.md 11742 chars
figures: ['fig1_arm_comparison.png', 'fig2_boundary_robustness.png', 'fig3_shap.png']
